# PKM_RF Triage Project Setup & Environment Verification (`setup.ipynb`)

Welcome to the **PKM_RF Emergency Severity Index (ESI) Triage Machine Learning System**.
This notebook verifies your environment (Python & R dependencies), validates project configuration, checks dataset availability, creates output directories, and provides execution triggers for all pipeline notebooks.

--- 
### Project Architecture & Notebook Inventory

1. **Configuration**: [`config/triage_conf.json`](file:///home/apt2736/PKM_RF/config/triage_conf.json)
   - Central project configuration defining dataset paths, target column (`esi`), train/val/test split ratios, random seed, and resampling ratios.

2. **Engineered Feature Distribution Plots**: [`models/plot_engineered.ipynb`](file:///home/apt2736/PKM_RF/models/plot_engineered.ipynb)
   - Generates distribution graphs for 26 feature-engineered inputs across ESI classes `1` through `5` in `plots/engineered_feature_distributions/`.

3. **3-Tier Hierarchical Training**: [`models/train_hierarchical_lightgbm_layers.ipynb`](file:///home/apt2736/PKM_RF/models/train_hierarchical_lightgbm_layers.ipynb)
   - Trains 4 LightGBM sub-models with layer-exclusive features (`deploy/*.rds`).

4. **Combined Master Pipeline Benchmark**: [`models/combined_hierarchical_triage_pipeline.ipynb`](file:///home/apt2736/PKM_RF/models/combined_hierarchical_triage_pipeline.ipynb)
   - Evaluates Soft Probabilistic Joint Product predictions vs Hard Case-Selector routing on the 1% holdout test set.

5. **Native C Transpilation**: [`models/transpile_soft_pipeline_to_c.ipynb`](file:///home/apt2736/PKM_RF/models/transpile_soft_pipeline_to_c.ipynb)
   - Transpiles all sub-models into zero-dependency C code (`deploy/triage_soft_pipeline.c`).

6. **Layer 1 Anomaly Detection Benchmark**: [`models/benchmark_layer1_anomaly_detectors.ipynb`](file:///home/apt2736/PKM_RF/models/benchmark_layer1_anomaly_detectors.ipynb)
   - Benchmarks One-Class SVM, Isolation Forest, K-Means, and LightGBM baseline.

7. **Automated Resampling Calibration**: [`models/calibrate_resampling_ratios_5fold_cv.ipynb`](file:///home/apt2736/PKM_RF/models/calibrate_resampling_ratios_5fold_cv.ipynb)
   - Optimizes Layer 1 class weight multipliers and Layer 2 resampling ratios using 5-Fold Stratified CV with Optuna.

In [ ]:
# ---------------------------------------------------------
# Step 1: Environment & Dependency Verification
# ---------------------------------------------------------
import os
import sys
import json
print("=== Python Environment Info ===")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Dir:    {os.getcwd()}")
required_py_pkgs = ['numpy', 'pandas', 'scipy', 'sklearn', 'lightgbm', 'optuna', 'matplotlib', 'seaborn', 'rpy2']
print("\nChecking Python Packages:")
for pkg in required_py_pkgs:
    try:
        __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
# Create required project output directories
for folder in ['deploy', 'reports', 'plots', 'plots/engineered_feature_distributions']:
    os.makedirs(folder, exist_ok=True)
    print(f"  [DIR OK] {folder}/")

In [ ]:
# ---------------------------------------------------------
# Step 2: Validate Configuration & Data Source File
# ---------------------------------------------------------
config_file = "config/triage_conf.json"
if os.path.exists(config_file):
    with open(config_file, "r") as f:
        config = json.load(f)
    print("\n=== Project Configuration Validated ===")
    print(f"  Data Source: {config['path']['data_source']}")
    print(f"  Target Col:  {config['classes']['target_col']}")
    
    data_path = config['path']['data_source']
    if os.path.exists(data_path):
        print(f"  [DATASET OK] Found data file: {data_path}")
    else:
        print(f"  [WARNING] Data file not found at: {data_path}")
else:
    print(f"[ERROR] Configuration file missing: {config_file}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Check R Environment & Required R Packages via rpy2
# ---------------------------------------------------------
try:
    %load_ext rpy2.ipython
    print("rpy2 extension loaded successfully.")
except Exception as e:
    print("Note on rpy2 initialization:", e)

In [ ]:
%%R
# Check R Packages
r_pkgs <- c("jsonlite", "caret", "dplyr", "ggplot2", "tidyr", "pROC", "lightgbm")
cat("Checking R Packages:\n")
for (pkg in r_pkgs) {
  if (suppressWarnings(require(pkg, character.only = TRUE, quietly = TRUE))) {
    cat(sprintf("  [OK] R package: %s\n", pkg))
  } else {
    cat(sprintf("  [MISSING] R package: %s\n", pkg))
  }
}

In [ ]:
# ---------------------------------------------------------
# Step 4: Pipeline Execution Helper
# Uncomment and run any notebook below to execute the full pipeline:
# ---------------------------------------------------------
# %run models/plot_engineered.ipynb
# %run models/train_hierarchical_lightgbm_layers.ipynb
# %run models/combined_hierarchical_triage_pipeline.ipynb
# %run models/transpile_soft_pipeline_to_c.ipynb
# %run models/calibrate_resampling_ratios_5fold_cv.ipynb
# %run models/benchmark_layer1_anomaly_detectors.ipynb
print("SETUP COMPLETED: Environment, Directories, and Configuration are ready!")